In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

In [8]:
# 데이터 디렉토리 설정
data_dir = "C:/Users/ko911/OneDrive/바탕 화면/UnsortedStudy/DNAage"



# AdditionalFile24NROMALIZATION.R.txt 파일 읽기
with open(f"{data_dir}/AdditionalFile24NROMALIZATION.R.txt", "r") as file:
    r_code = file.read()

# R 코드에서 주석 및 불필요한 내용 제거
r_code = r_code.replace("#", "").replace("\n\n", "\n")

# trafo 함수 정의
def trafo(x, adult_age=20):
    x = (x + 1) / (1 + adult_age)
    y = np.where(x <= 1, np.log(x), x - 1)
    return y

# anti.trafo 함수 정의
def anti_trafo(x, adult_age=20):
    y = np.where(x < 0, (1 + adult_age) * np.exp(x) - 1, (1 + adult_age) * x + adult_age)
    return y

# AdditionalFile22probeAnnotation21kdatMethUsed.csv 파일 읽기
probeAnnotation21kdatMethUsed = pd.read_csv(f"{data_dir}/AdditionalFile22probeAnnotation21kdatMethUsed.csv")

# AdditionalFile21datMiniAnnotation27k.csv 파일 읽기
probeAnnotation27k = pd.read_csv(f"{data_dir}/AdditionalFile21datMiniAnnotation27k.csv")

# AdditionalFile23predictor.csv 파일 읽기
datClock = pd.read_csv(f"{data_dir}/AdditionalFile23predictor.csv")

# DNA 메틸화 데이터 (beta 값) 읽기
dat0 = pd.read_csv(f"{data_dir}/AdditionalFile26MethylationDataExample55.csv")
n_samples = dat0.shape[1] - 1
n_probes = dat0.shape[0]
dat0.iloc[:, 0] = dat0.iloc[:, 0].str.replace("\"", "")

# 로그 파일 생성 및 오류 체크
with open("LogFile.txt", "w") as log_file:
    log_file.write(f"메틸화 데이터에는 {n_samples}개의 샘플 (배열)과 {n_probes}개의 프로브가 있습니다.\n")
    if n_samples == 0:
        log_file.write("\n ERROR: 샘플이 없는 것 같습니다. 데이터 파일을 올바르게 입력했는지 확인하세요.\n")
    if n_probes == 0:
        log_file.write("\n ERROR: 프로브가 없는 것 같습니다. 데이터 파일을 올바르게 입력했는지 확인하세요.\n")



In [10]:
# Initialize DoNotProceed
DoNotProceed = False

# Check for more samples than CpG probes
if n_samples > n_probes:
    with open("LogFile.txt", "a") as log_file:
        log_file.write("\n MAJOR WARNING: It worries me a lot that there are more samples than CpG probes.\n Make sure that probes correspond to rows and samples to columns.\n I wonder whether you want to first transpose the data and then resubmit them? In any event, I will proceed with the analysis.")

# Check if the first column contains numeric values
if dat0.iloc[:, 0].dtype == "float64":
    DoNotProceed = True
    with open("LogFile.txt", "a") as log_file:
        log_file.write(f"\n Error: The first column does not seem to contain probe identifiers (cg numbers from Illumina) since these entries are numeric values. Make sure that the first column of the file contains probe identifiers such as cg00000292. Instead it contains {dat0.iloc[0:3, 0].to_list()}")

# Check if the first column contains character values
if dat0.iloc[:, 0].dtype != "object":
    with open("LogFile.txt", "a") as log_file:
        log_file.write(f"\n Major Warning: The first column does not seem to contain probe identifiers (cg numbers from Illumina) since these entries are numeric values. Make sure that the first column of the file contains CpG probe identifiers such as cg00000292. Instead it contains {dat0.iloc[0:3, 0].to_list()}")

# Create a DataFrame for datout
datout = pd.DataFrame({
    "Error": ["Input error. Please check the log file for details", "Please read the instructions carefully."],
    "Comment": ["", "email Steve Horvath."]
})

if not DoNotProceed:
    non_numeric_column = ~dat0.iloc[:, 1:].apply(pd.api.types.is_numeric_dtype)
    if non_numeric_column.sum() > 0:
        with open("LogFile.txt", "a") as log_file:
            log_file.write("\n MAJOR WARNING: Possible input error. The following samples contain non-numeric beta values: "
                           f"{', '.join(dat0.columns[1:][non_numeric_column])}\n Hint: Maybe you use the wrong symbols "
                           "for missing data. Make sure to code missing values as NA in the Excel file. To proceed, "
                           "I will force the entries into numeric values but make sure this makes sense.\n")

    X_chromosomal_CpGs = probe_annotation_27k[probe_annotation_27k["Chr"] == "X"]["Name"].tolist()
    select_X_chromosome = dat0.iloc[:, 0].isin(X_chromosomal_CpGs)
    select_X_chromosome[select_X_chromosome.isna()] = False
    mean_X_chromosome = pd.Series(index=dat0.columns[1:], dtype=float)
    
    if select_X_chromosome.sum() >= 500:
        mean_X_chromosome = dat0.loc[select_X_chromosome, dat0.columns[1:]].apply(pd.to_numeric).mean(numeric_only=True)
    
    if mean_X_chromosome.isna().sum() > 0:
        with open("LogFile.txt", "a") as log_file:
            log_file.write("\n \n Comment: There are lots of missing values for X chromosomal probes for some of the samples. "
                           "This is not a problem when it comes to estimating age but I cannot predict the gender of these samples.\n")


In [11]:
# 단계 2: 21k 프로브로 데이터 제한 및 숫자 형식 확인
match1 = pd.Series([dat0.iloc[:, 0].tolist().index(name) if name in dat0.iloc[:, 0].tolist() else np.nan for name in probeAnnotation21kdatMethUsed['Name']])
if match1.isna().sum() > 0:
    raise Exception(f"{match1.isna().sum()} CpG probes cannot be matched")
dat1 = dat0.iloc[match1.dropna().astype(int), :]
asnumeric1 = lambda x: pd.to_numeric(x, errors='coerce')
dat1.iloc[:, 1:] = dat1.iloc[:, 1:].applymap(asnumeric1)

# 단계 3: 결과 파일 datout 생성
np.random.seed(1)
# 데이터 정규화 수행 여부 (권장)
normalizeData = True
# AdditionalFile25StepwiseAnalysis.txt 파일을 포함하고 있는 코드를 여기에 삽입

# 단계 4: 결과 출력
if (datout["Comment"] == "").all():
    with open("LogFile.txt", "a") as log_file:
        log_file.write("\n 개별 샘플은 정상적으로 처리되었습니다.")
if (datout["Comment"] != "").any():
    with open("LogFile.txt", "a") as log_file:
        log_file.write(f"\n 경고: 다음 샘플에 대해 경고가 생성되었습니다.\n {', '.join(datout.loc[datout['Comment'] != '', 'Error'])}\n 자세한 내용은 로그 파일을 확인하세요.")

# 결과를 디렉토리에 출력
datout.to_csv("Output.csv", index=False, sep=",")
#위 코드에서 normalizeData = True 다음에 AdditionalFile25StepwiseAnalysis.txt 파일을 포함하고 있는 부분은 해당 파일의 내용을 적절히 읽어와 실행하셔야 합니다. 코드를 실행하기 전에 주의깊게 원래 R 코드와 비교하며 변수 및 데이터프레임 이름, 함수 등을 올바르게 대응시켜 주시기 바랍니다.


C:\Users\ko911\AppData\Local\Temp\ipykernel_15384\2702424073.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dat1.iloc[:, 1:] = dat1.iloc[:, 1:].applymap(asnumeric1)
